# From Backpropagation to Predictive Coding

*A hands-on introduction for a Master's thesis on physics-based hardware for Predictive Coding.*

This notebook takes you from a standard MLP trained with **backpropagation** — the software gold standard — to **Predictive Coding (PC)**, a local, energy-based learning algorithm that is a natural fit for analog hardware.

By the end you will be able to:

- write the backward pass of a small MLP by hand, and explain what `.backward()` really does;
- explain the three structural downsides of backprop (activation memory, layer locking, global error transport) and *why they hurt physical implementations*;
- explain PC: states, predictions, prediction errors, the energy function, and the two-phase learning loop;
- implement PC in PyTorch yourself, verify it against an exact analytical optimum, and train it on a real classification task;
- explain why backprop suits von-Neumann machines and why PC suits analog hardware.

**How to use this notebook.** It is organised in chapters (Ch 0–4) that can be done separately.

- **Exercises** are marked `# === EXERCISE ===`; each is followed by a small test that checks your solution.
- **Think** questions ask you to reason before reading on — each comes with a hidden *hint* and a hidden *answer*.
- Cells that are only there for **plotting** are marked with a comment at the top (`# --- plotting: no need to read ---`); you can skip reading them.

**The problem.** Binary classification on the *two-moons* dataset: two interleaved half-moons of points, each moon a class, and we want to learn which moon a point belongs to.

| Chapter | Question it answers |
|---|---|
| Ch 0 · Setup | What task should the model learn? |
| Ch 1 · MLPs & backprop | How does a neural network learn, and what does `.backward()` do internally? |
| Ch 2 · PC theory | What is Predictive Coding, in continuous time? |
| Ch 3 · PC in PyTorch | How do we turn PC into working software? |
| Ch 4 · Takeaways | Why does this matter for the thesis? |

# Ch 0 · Setup & the playground

**What we will explain here:** the dataset, the task, and what a good solution looks like.

We use a tiny dataset so that everything runs fast and we can look at everything. The dataset is called **two-moons**: two interleaved half-moons of points, each moon a different class. The input of our model will be the 2D coordinates of a point; the output will be its class.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from typing import List, Optional

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
def create_moons_dataset(n_samples=500, noise=0.1, test_size=0.2):
    """Create the two-moons dataset for binary classification."""
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    X_train = torch.FloatTensor(X_train)
    X_test = torch.FloatTensor(X_test)
    y_train = torch.FloatTensor(y_train).unsqueeze(1)
    y_test = torch.FloatTensor(y_test).unsqueeze(1)
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = create_moons_dataset()
print(f"Train set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:  {X_test.shape[0]} samples")

In [ ]:
# --- plotting: no need to read ---
plt.figure(figsize=(6, 5))
plt.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train[:, 0],
    cmap="RdYlBu",
    edgecolors="black",
    linewidth=0.5,
    s=30,
)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Two-moons dataset (train)")
plt.show()

**Think.**

1. Can a straight line perfectly separate these two moons?

<details><summary>Hint</summary>Look at how the two arcs are arranged: a straight line can separate points that sit on opposite sides of it. Do the two moons sit on opposite sides of any single line?</details>

<details><summary>Answer</summary>No. The two moons curve around each other, so any straight line leaves some points of each class on the wrong side.</details>

2. If a model could only draw straight lines, what would its best attempt look like?

<details><summary>Hint</summary>It can only choose one line; it wants to minimise how many points end up on the wrong side of it.</details>

<details><summary>Answer</summary>It would pick the line that misclassifies the fewest points — roughly a diagonal cut through the middle — and simply accept the errors. That is exactly what a linear model learns to do, and why we will need nonlinearities later.</details>

# Ch 1 · MLPs & backpropagation

**What we will explain here:** how an MLP's forward pass works, what `.backward()` does internally (you will re-implement it by hand), and the three structural downsides of backprop.

## 1.1 The forward pass — activations in your hands

We use a small MLP with architecture `[2, 16, 16, 1]`: 2 inputs, two hidden layers of 16 neurons, 1 output.

- the hidden layers use **tanh** activations;
- the output uses a **sigmoid** — a number in (0,1), our prediction $\hat y$, interpreted as "probability of class 1";
- the loss is the (summed) mean squared error $L = \tfrac{1}{2}\sum_b (\hat y_b - y_b)^2$.

Written out, with weight matrices $W_1, W_2, W_3$ and biases $b_1, b_2, b_3$:

$$h_1 = \tanh(W_1 x + b_1), \qquad h_2 = \tanh(W_2 h_1 + b_2), \qquad \hat y = \sigma(W_3 h_2 + b_3), \qquad L = \tfrac{1}{2}\|\hat y - y\|^2 .$$

The forward pass runs left to right. We will store, for every layer, the **pre-activation** $z_i$ and the **activation** $h_i$. Why? Because the backward pass (next section) needs them *again* — and as we'll see in §1.4, that storage is exactly what makes backprop memory-hungry.

**Think.** Why use a sigmoid on the output rather than a tanh?

<details><summary>Hint</summary>Compare the range of values each activation can output, and the range of our targets (0/1).</details>

<details><summary>Answer</summary>A sigmoid maps to (0,1), which matches the 0/1 targets; a tanh maps to (−1,1), so you would have to relabel the targets to ±1.</details>

In [ ]:
class MLP(nn.Module):
    """A small MLP with tanh hidden layers and a sigmoid output."""

    def __init__(self, layer_sizes: List[int] = [2, 16, 16, 1]):
        super().__init__()
        self.layer_sizes = layer_sizes
        self.layers = nn.ModuleList()
        for i in range(len(layer_sizes) - 1):
            self.layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
        self.cache = None  # forward pass will store (x, (z1,h1), (z2,h2), (z3, y_hat))
        self.grads = None  # the hand-computed gradients will go here

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass. Stores every (pre-activation, activation) in self.cache."""
        self.cache = [x]
        h = x
        for i, layer in enumerate(self.layers):
            z = layer(h)  # pre-activation
            h = torch.tanh(z) if i < len(self.layers) - 1 else torch.sigmoid(z)  # activation
            self.cache.append((z, h))
        return h

    def loss(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        y_hat = self.forward(x)
        return 0.5 * torch.sum((y_hat - y) ** 2)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward(x)

In [ ]:
# How much does the forward pass have to store?
mlp = MLP()
_ = mlp.forward(X_train[:32])
print("Cached during the forward pass:")
for i, (z, h) in enumerate(mlp.cache[1:], start=1):
    print(f"  layer {i}: pre-activation {tuple(z.shape)}, activation {tuple(h.shape)}")

## 1.2 Training with "magic" `.backward()`

Here is the black box. `loss.backward()` computes the gradient of the loss with respect to every parameter, and `optimizer.step()` nudges the parameters downhill. We will not question *how* it works yet — first, let's just look: it works.

In [ ]:
# --- plotting: no need to read ---
def plot_decision_boundary(model, X, y, ax=None, title=None):
    """Plot a model's decision boundary over the data (expects model.predict(x))."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = torch.FloatTensor(np.stack([xx.ravel(), yy.ravel()], axis=1))
    with torch.no_grad():
        Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlBu", levels=20)
    ax.scatter(X[:, 0], X[:, 1], c=y[:, 0], cmap="RdYlBu", edgecolors="black", linewidth=0.5, s=30)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    if title:
        ax.set_title(title)
    return ax

In [ ]:
def train_mlp(epochs=100, batch_size=32, lr=0.1):
    """Train the MLP with (magic) backprop. Returns (model, losses, accs)."""
    torch.manual_seed(0)
    model = MLP()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    losses, accs = [], []

    for epoch in range(epochs):
        perm = torch.randperm(len(X_train))
        epoch_loss = 0.0
        n_batches = 0
        for i in range(0, len(X_train), batch_size):
            idx = perm[i : i + batch_size]
            xb, yb = X_train[idx], y_train[idx]

            optimizer.zero_grad()
            loss = model.loss(xb, yb)
            loss.backward()  # the magic
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        losses.append(epoch_loss / n_batches)
        with torch.no_grad():
            acc = ((model.forward(X_test) > 0.5).float() == y_test).float().mean().item()
        accs.append(acc)

    return model, losses, accs


mlp_model, mlp_losses, mlp_accs = train_mlp()
print(f"Final test accuracy: {mlp_accs[-1] * 100:.1f}%")

In [ ]:
# --- plotting: no need to read ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(mlp_losses)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Backprop: training loss")
axes[1].plot(mlp_accs)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test accuracy")
axes[1].set_title("Backprop: test accuracy")
plt.tight_layout()
plt.show()

plot_decision_boundary(mlp_model, X_train, y_train, title="Backprop: decision boundary")
plt.show()

**Think.** The input, the architecture and the activation functions were fixed. What did the optimizer actually change during training — and therefore, where does the network store what it has learned?

<details><summary>Hint</summary>List everything the training loop modifies between the start and the end of training.</details>

<details><summary>Answer</summary>Only the parameters (weights and biases) changed. The learned mapping is stored entirely in the weights — which is why the mystery we solve next (how each weight "knows" how to change) matters so much.</details>

## 1.3 What does `.backward()` do? — backprop by hand

The loss $L$ is a function of the weights through a chain of functions. The **chain rule** lets us get gradients for layers deeper down, by propagating errors **backwards**, layer by layer.

Define the error signals (standard textbook notation):

$$\delta_3 = (\hat y - y) \odot \sigma'(z_3), \qquad \delta_2 = (W_3^\top \delta_3) \odot \tanh'(z_2), \qquad \delta_1 = (W_2^\top \delta_2) \odot \tanh'(z_1),$$

with $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$ and $\tanh'(z) = 1 - \tanh^2(z)$. The weight gradients then are

$$\frac{\partial L}{\partial W_3} = \delta_3\, h_2^\top, \qquad \frac{\partial L}{\partial W_2} = \delta_2\, h_1^\top, \qquad \frac{\partial L}{\partial W_1} = \delta_1\, x^\top,$$

and $\partial L/\partial b_i = \sum_{\text{batch}} \delta_i$.

Implement it yourself — using only plain tensor arithmetic, no autograd.

In [ ]:
# === EXERCISE ===
def backward_manual(self, y: torch.Tensor) -> None:
    """Hand-written backward pass for THIS MLP.

    Given the target y, compute the gradient of the loss
        L = 0.5 * sum((y_hat - y)^2)
    w.r.t. every parameter, using only the cached activations. Store them as a
    list of dicts in self.grads:
        self.grads = [
            {"W": dL/dW_1, "b": dL/db_1},
            {"W": dL/dW_2, "b": dL/db_2},
            {"W": dL/dW_3, "b": dL/db_3},
        ]

    Work through the chain rule exactly as derived in section 1.3: you will
    need the cached pre-activations (for the activation derivatives) and the
    cached activations (for the weight gradients). No autograd allowed here.
    """
    x = self.cache[0]
    z1, h1 = self.cache[1]
    z2, h2 = self.cache[2]
    z3, y_hat = self.cache[3]

    # YOUR CODE HERE
    raise NotImplementedError


# attach the method to the class (given — do not remove)
MLP.backward_manual = backward_manual

In [ ]:
# Test: does your hand-written backward pass match autograd?
torch.manual_seed(1)
mlp = MLP()
xb, yb = X_train[:32], y_train[:32]

loss = mlp.loss(xb, yb)
loss.backward()  # the oracle
mlp.backward_manual(yb)  # your implementation

ok = True
for i, layer in enumerate(mlp.layers):
    ok &= torch.allclose(mlp.grads[i]["W"], layer.weight.grad, atol=1e-5)
    ok &= torch.allclose(mlp.grads[i]["b"], layer.bias.grad, atol=1e-5)

if ok:
    print("✓ test passed — your manual backprop matches autograd!")
else:
    print("✗ mismatch — compare your grads to the autograd ones (layer.weight.grad).")

**Think.**

1. Where does the global loss first "enter" the backward pass?

<details><summary>Hint</summary>Which $\delta$ is defined directly from $(\hat y - y)$, and which ones are computed from the others?</details>

<details><summary>Answer</summary>At the output layer: $\delta_3 = (\hat y - y)\,\sigma'(z_3)$ is the only error that uses the loss directly; every other $\delta$ is built from it.</details>

2. What does that imply for a neuron deep in the network?

<details><summary>Hint</summary>What information must physically reach that neuron before it can update itself?</details>

<details><summary>Answer</summary>It can only update once the global loss signal has been transported all the way back to it, layer by layer. It cannot derive its update from local information alone.</details>

## 1.4 Why backprop is awkward

Three structural costs — all of which you've now seen with your own eyes:

1. **Memory for activations.** The forward pass must *store* every activation (look at `self.cache`!). For deep networks this dominates memory. On a chip, that means on-chip storage.
2. **Layer locking / synchronisation.** The backward pass can only start *after* the forward pass completes, and it runs strictly in reverse layer order. Every layer is idle while the others compute; updates cannot be pipelined. There is an implied global clock / schedule.
3. **Global error transport.** Every neuron's update is $-\eta\,\partial L/\partial W_i$, obtained by propagating the *global* loss backward. No neuron can update itself from local information. **Why this matters physically:** in a real physical system — a brain or a chip — a single neuron cannot know the global loss. Communicating it everywhere would require dedicated wiring and energy just to *transport* a signal that is fundamentally global.

**Think.**

1. Which of these three can the brain do, and which can it not do?

<details><summary>Hint</summary>Run backprop's three requirements as a checklist against what a biological neuron physically has. Does the brain keep activations around forever? Does it globally synchronise all of its layers? Does a global loss travel backwards through it?</details>

<details><summary>Answer</summary>None of the three. The brain cannot hold activations in memory indefinitely (it has to move on to the next input), it cannot globally synchronise its layers (they are unclocked and local), and it cannot transport a precise global error signal backwards (there is no reverse wiring for it). All three structural requirements of backprop are things the brain does not do.</details>

2. What does that tell you about training in analog hardware?

<details><summary>Hint</summary>Analog hardware is much closer to a brain than to a von-Neumann computer: no clock, no big shared memory, no global broadcast.</details>

<details><summary>Answer</summary>All of these are problems for analog hardware, the same way they are for the brain — after all, the brain *is* analog. Predictive Coding is a learning algorithm that comes from neuroscience, so it is a much better fit for analog hardware than backprop.</details>

# Ch 2 · Predictive Coding — the theory

**What we will explain here:** the central idea (relaxing the strict feedforward constraint), the energy function, and the two-phase learning loop that makes PC tick. We keep everything in continuous time until §2.4.

## 2.1 The core idea: from strict feedforward to slack

In the MLP of Ch 1, every layer is strictly determined by the layer below it: the output of layer $i$ *is* the input of layer $i+1$,

$$h_{i+1} = f_{i+1}(h_i),$$

with no room for disagreement. The activations are fixed once the input is fixed; the only free variables in the whole network are the weights. That is exactly what makes backprop's job hard: to change anything, the *global* loss has to be transported all the way back through these rigid layers.

Predictive Coding starts from a simple relaxation of this constraint. We give each layer its own **free state** $s_i$, and we let each layer **predict** the state above it:

$$\hat s_{i+1} = f_{i+1}(s_i).$$

The state $s_{i+1}$ is now *allowed* to deviate from its prediction. That deviation is the slack — the **prediction error**:

$$\varepsilon_{i+1} = s_{i+1} - \hat s_{i+1}.$$

If the network is behaving well, the layers agree and the slack is small; if not, the slack is large. The goal of PC is to make the layers maximally agree, minimizing the slack. (both by updating the states and later also the weights)

The output works the same way: the output node's target is $y$, the final layer predicts $\hat y = f_L(s_{L-1})$, and its slack is $\varepsilon_L = y - \hat y$. This ties closely to the classification loss $L = \tfrac{1}{2}\|\hat y - y\|^2$ we already know from Ch 1. Here, it's not a special term, it's just one more term in the network.

Collecting all of the slack gives the **energy** of the network:

$$E = \tfrac{1}{2}\sum_{i=1}^{L} \|\varepsilon_i\|^2 .$$

_(We call it "energy", because the PC system wants to minimize this quantity, just like a physical system would)_

**Locality.** Every term involves only *neighbouring* quantities: node $i$'s energy depends only on $s_i$ and $s_{i-1}$ (via $\hat s_i$) and on $s_{i+1}$ (via the term above).

**Notation translation table.** PC reuses the same building blocks as backprop, with different names. Keep this table at hand:

| Backpropagation | Predictive Coding |
|---|---|
| loss $L(\hat y, y)$ (global) | energy $E = \tfrac{1}{2}\sum_i \|\varepsilon_i\|^2$ (sum of local terms) |
| activations $a_i$ (determined by the forward pass) | states $s_i$ (free variables) |
| forward pass computes $a_i$ | predictions $\hat s_{i+1} = f_{i+1}(s_i)$ |
| error signal $\delta_i$ (backpropagated from loss) | prediction error $\varepsilon_i = s_i - \hat s_i$ (local) |
| backward pass, reverse order, global | top-down errors, layer-local, simultaneous |
| weight update $\Delta W_i = -\eta\,\delta_i a_{i-1}^\top$ | weight update $\Delta W_i = -\eta\,\varepsilon_i s_{i-1}^\top$ (same local form!) |
| error computed by backprop through all layers | error emerges from energy settling |

**Think.**

1. The weight update has the same local shape in both algorithms — $\delta_i a_{i-1}^\top$ vs $\varepsilon_i s_{i-1}^\top$. What is the *only* difference?

<details><summary>Hint</summary>Look at the two error symbols in the table — $\delta$ and $\varepsilon$. What does each one require in order to exist at layer $i$?</details>

<details><summary>Answer</summary>Only the origin of the error signal: backprop computes $\delta_i$ by transporting the global loss backward; PC gets $\varepsilon_i$ locally from energy settling. The update formula itself is identical.</details>

2. Imagine you are a single neuron in some network. How would you know how you affect the final prediction — (a) in backprop, (b) in PC?

<details><summary>Hint</summary>In (b), a neuron can only measure what is at its own location: its state and its neighbours' states. In (a), where must the relevant information come from, and how far does it have to travel?</details>

<details><summary>Answer</summary>In backprop, you receive $\delta$ from above: the global loss is transported back to you, layer by layer. In PC, you never need the global loss. The "global" information reaches you *indirectly*: the inference loop (next section) spreads it through purely local interactions — your neighbours' errors are shaped by *their* neighbours, and so on — and by the time the states settle, the effect of the loss has propagated everywhere without ever being broadcast. Thanks to that, your own weight update stays fully local: it uses only your error $\varepsilon_i$ and your input $s_{i-1}$.</details>

## 2.2 The two-phase learning loop

Training a PC network alternates two phases. This is the heart of the algorithm, so let's take it slowly.

### Phase 1 — inference: let the states settle

During inference we **hold the weights fixed** and let the *states* move, so that the energy decreases. The rule each state follows is simple: move in the direction that lowers $E$,

$$\frac{ds_i}{dt} = -\frac{\partial E}{\partial s_i} .$$

(Why *this* rule? It guarantees the energy can only go down — see the side-note later.)

Expanding the right-hand side shows that each state feels exactly **two local forces**:

$$\frac{ds_i}{dt} = \underbrace{-\varepsilon_i}_{\text{bottom-up}} \;+\; \underbrace{\left(\frac{\partial \hat s_{i+1}}{\partial s_i}\right)^\top \varepsilon_{i+1}}_{\text{top-down}} .$$

- **Bottom-up** ($-\varepsilon_i$): the state is pulled toward its own prediction $\hat s_i$. If layer $i$ is not predicting itself well, this force corrects it.
- **Top-down** ($+(\partial \hat s_{i+1}/\partial s_i)^\top \varepsilon_{i+1}$): the state is nudged so that the layer *above* becomes more predictable. The Jacobian $(\partial \hat s_{i+1}/\partial s_i)^\top$ says "if changing $s_i$ slightly would make $\hat s_{i+1}$ match $s_{i+1}$ better, do it."

So a state is squeezed between its own prediction error and the prediction error of the layer above. It stops moving when the two forces balance, i.e. when $\partial E/\partial s_i \approx 0$ — the states have **settled**.

What has the network done by settling? Before inference, the states were arbitrary (say, zero). After settling, every layer has adjusted itself so that it both predicts the layer below and is predicted by the layer above, *given this particular data point*. The states have "configured themselves" in response to the data and the current weights.

**Think.** Why must the states settle *before* we touch the weights?

<details><summary>Hint</summary>The weight update (below) uses the prediction errors $\varepsilon_i$. What are those errors worth before the states have adjusted to the data?</details>

<details><summary>Answer</summary>Before settling, the states are arbitrary, so the errors don't reflect the data or the weights; updating weights on un-settled errors would be a near-random step. Settling makes the errors meaningful (the states are "as consistent as possible"), so the weight update moves in a useful direction.</details>

### Phase 2 — weight update

With the states settled, we hold *them* fixed and take one gradient step on the weights:

$$\Delta W_i = -\eta\,\frac{\partial E}{\partial W_i} = -\eta\,\varepsilon_i\, s_{i-1}^\top .$$

Layer $i$ updates its weights using only two local quantities: its own prediction error $\varepsilon_i$ and its own input $s_{i-1}$. No signal from the top of the network is involved.

This is **prospective configuration**: the states configure themselves *first* (anticipating what the weights should do), and the weights follow the states. Backprop is the opposite — activities are fixed by the forward pass, and only the error moves backward.

The two phases alternate for every training example: settle the states (weights fixed), update the weights (states fixed), move on. Notice that each phase only ever touches local quantities — that locality is what we will exploit in software and, eventually, in hardware.

> **Side-note**: why `ds/dt = -dE/ds`?
>
> The state rule above was a *choice*. A natural question is: is it a good one? It is — here is the one-line reason.
>
> Along the flow $ds_i/dt = -\partial E/\partial s_i$, the energy changes at rate
>
> $$\frac{dE}{dt} = \sum_i \frac{\partial E}{\partial s_i}\frac{ds_i}{dt} = -\sum_i \Big\|\frac{\partial E}{\partial s_i}\Big\|^2 \le 0 .$$
>
> The energy can only go downhill, and it stops exactly at an equilibrium of the forces ($\partial E/\partial s_i = 0$). So this single rule *guarantees* that inference reaches a minimum of the energy. *(For the curious: $E$ is a Lyapunov function of the state dynamics.)* This is reassuring, but it is not the part of PC that matters most — the *locality* of everything above is.

## 2.3 From continuous time to software

A computer is a discrete, clocked machine: it cannot literally run a continuous differential equation. Every continuous system we simulate has to be *discretised*. For us the translation is almost trivial:

- the continuous rule $ds_i/dt = -\partial E/\partial s_i$ says "each state flows downhill in $E$";
- downhill flow, in small steps, is exactly what **gradient descent** does;
- so in software we simply treat $E$ as a loss and the states as parameters, and minimise $E$ with the standard `torch.optim.SGD` — the same tool we used to train the MLP in Ch 1.

The only choices we make are the discretisation analogues:

| Continuous time | Software |
|---|---|
| time step $dt$ | learning rate (`s_lr`) |
| "until settled" | a fixed number of steps (`iters`) or a convergence check |

## 2.4 Getting started: linear PC models

For simplicity, let's start with **linear** models (no activations): $\hat s_{i+1} = W_{i+1}s_i + b_{i+1}$.

Also, lucky for us, a linear PC network has a **nice analytical formula** for the energy minimum: we know exactly what the energy should be once the states have settled. That gives us a ground truth to check our inference code against. (No need to derive it — it is implemented for us in `analytic_E` below.)

The class below is given, except for the pieces you will implement. Note that `energy` must also store the prediction errors in `self.errors` — they are needed by `local_state_grad` and by `update_weights`.

In [ ]:
class LinearPC:
    """Linear Predictive Coding network (no activations)."""

    def __init__(self, layer_sizes: List[int], s_lr: float = 0.2, w_lr: float = 0.01):
        self.layer_sizes = layer_sizes
        self.s_lr = s_lr  # state (inference) learning rate
        self.w_lr = w_lr  # weight learning rate
        self.layers = nn.ModuleList()
        for i in range(len(layer_sizes) - 1):
            self.layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
        self.states: Optional[List[torch.Tensor]] = None
        self.errors: Optional[List[torch.Tensor]] = None

    def parameters(self):
        """All trainable parameters (the layer weights and biases)."""
        for layer in self.layers:
            yield from layer.parameters()

    def y_hat(self, x: torch.Tensor) -> torch.Tensor:
        """Feedforward prediction (used for testing / readout)."""
        s = x
        for layer in self.layers:
            s = layer(s)
        return s

    def analytic_E(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Closed-form optimum of the energy for a LINEAR network:

            E* = 0.5 * sum_b (y_b - y_hat_ff,b)^T S^-1 (y_b - y_hat_ff,b)
            S  = I + sum_{l=2}^{L} W_{L:l} W_{L:l}^T

        with y_hat_ff the feedforward output. Used only as a ground truth.
        """
        Ws = [l.weight.detach() for l in self.layers]
        L = len(Ws)
        S = torch.eye(y.shape[1])
        for ell in range(2, L + 1):
            P = Ws[L - 1]
            for k in range(L - 1, ell - 1, -1):
                P = P @ Ws[k - 1]
            S = S + P @ P.T
        d = y - self.y_hat(x)
        return 0.5 * torch.einsum("ni,ij,nj->", d, torch.linalg.inv(S), d)

**Exercise 1 — `init_states`.** The class does not yet know how to set up its states. Implement it: `s_0` is pinned to the input; every other state starts at zero. By making this an explicit method now, we will later be able to simply *override* it.

In [ ]:
# === EXERCISE ===
def init_states(self, x: torch.Tensor) -> None:
    """Initialise the free states and store them in self.states.

    Args:
        x: input tensor of shape (batch, n_input)

    Rules:
        - s_0 is the input x itself (do NOT add requires_grad to it);
        - every other state s_i (i >= 1) is a zero tensor of shape
          (batch, layer_sizes[i]) with requires_grad=True.
    """
    self.states = [x]
    for size in self.layer_sizes[1:-1]:
        # YOUR CODE HERE
        raise NotImplementedError


LinearPC.init_states = init_states

In [ ]:
# Test: init_states
torch.manual_seed(2)
pc = LinearPC([2, 8, 1])
x = X_train[:4]
pc.init_states(x)

assert pc.states[0] is x, "s_0 must be the input itself"
assert len(pc.states) == 2, f"expected states [s_0, s_1], got {len(pc.states)}"
s1 = pc.states[1]
assert tuple(s1.shape) == (4, 8), f"s_1 shape should be (4, 8), got {tuple(s1.shape)}"
assert s1.requires_grad, "free states must require grad"
assert torch.allclose(s1, torch.zeros(4, 8)), "free states must start at zero"
print("✓ test passed — init_states OK")

**Exercise 2 — `energy`.** Implement the energy: the sum over all prediction-error energies, including the output term $\tfrac{1}{2}\|y - \hat y\|^2$. Also store the prediction errors in `self.errors` — they are used by `local_state_grad` and by `update_weights` below.

The states are *independent* tensors — `init_states` created them without any dependence on the weights. That independence is exactly what keeps the weight update local later: gradients can never flow from the weights through the states. (The tool that enforces such independence is `.detach()`; you'll see it in Ch 3.)

In [ ]:
# === EXERCISE ===
def energy(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """The energy E = 0.5 * sum_i ||eps_i||^2 + 0.5 * ||y - y_hat||^2.

    Loop over the layers, accumulate the squared prediction errors, and
    store the errors in self.errors (used by local_state_grad and by
    update_weights).

    The states are independent tensors (created in init_states) — treat
    them as the fixed inputs of this function.
    """
    # YOUR CODE HERE
    raise NotImplementedError


LinearPC.energy = energy

In [ ]:
# Test: energy matches an independent reference, and the errors are stored.
torch.manual_seed(6)
pc = LinearPC([2, 8, 1])
x, y = X_train[:4], y_train[:4]
pc.init_states(x)
E = pc.energy(x, y)

ref = 0.0
ref_errors = []
for i in range(len(pc.layers) - 1):
    pred = pc.layers[i](pc.states[i])
    err = pc.states[i + 1] - pred
    ref_errors.append(err)
    ref += 0.5 * torch.sum(err**2)
y_hat = pc.layers[-1](pc.states[-1])
err_out = y - y_hat
ref_errors.append(err_out)
ref += 0.5 * torch.sum(err_out**2)

ok_energy = torch.allclose(E, ref)
ok_errors = len(pc.errors) == len(pc.layers) and all(
    e.shape == r.shape for e, r in zip(pc.errors, ref_errors)
)
print(f"energy matches reference  : {ok_energy}")
print(f"errors stored & shaped    : {ok_errors}")
if ok_energy and ok_errors:
    print("✓ test passed — energy OK")
else:
    print("✗ check your energy implementation")

**Exercise 3 — the single-layer backward pass, by hand.** Because every term of $E$ is local, the gradient of $E$ w.r.t. one state depends only on that state's immediate neighbourhood. Implement `local_state_grad` for one layer, *analytically* — plain tensor arithmetic, no autograd. Signs matter: bottom-up minus top-down.

Hint: use the online matrix-calculus tool https://www.matrixcalculus.org/ . You can enter something like `0.5*norm2(s_next-W*s_i)^2` to get the derivative of one energy term w.r.t. `s_i` (this is basically the energy function written in the tool's API). Note that our tensors are in (batch, features) row layout, so the linear-algebra expression $W^\top \varepsilon$ is written `eps @ W` here.

In [ ]:
# === EXERCISE ===
def local_state_grad(self, state, pred, W_next, err_next):
    """Gradient of the energy w.r.t. one state, from LOCAL info only.

    Only two energy terms touch s_i: its own prediction error (bottom-up)
    and the prediction error of the layer above (top-down). Combine them
    with the right signs — section 2.2 shows the decomposition.

    Args:
        state:    s_i (the state whose gradient we want)
        pred:     s_hat_i = f_i(s_{i-1})   (this layer's prediction of s_i)
        W_next:   weight matrix of the layer above, W_{i+1}
        err_next: prediction error at the layer above, eps_{i+1}
                  (= s_{i+1} - s_hat_{i+1}, or y - y_hat for the output)

    Returns:
        dE/ds_i as a tensor of the same shape as state.
    """
    # YOUR CODE HERE
    raise NotImplementedError


LinearPC.local_state_grad = local_state_grad

In [ ]:
# Test: does your local gradient match autograd?
torch.manual_seed(3)
pc = LinearPC([2, 8, 1])
xb, yb = X_train[:8], y_train[:8]

pc.init_states(xb)
E = pc.energy(xb, yb)
E.backward()

ok = True
for j in range(1, len(pc.states)):
    state = pc.states[j]
    pred = pc.layers[j - 1](pc.states[j - 1])
    manual = pc.local_state_grad(state, pred, pc.layers[j].weight, pc.errors[j])
    ok &= torch.allclose(manual, state.grad, atol=1e-4)

if ok:
    print("✓ test passed — your local gradient matches autograd!")
else:
    print("✗ mismatch — compare your result to state.grad.")

**Exercise 4 — `inference`.** Now assemble Phase 1. Here is the key trick that makes it easy: the energy $E$ is *just another loss*, and the states are *just another set of parameters*. So settling the states is nothing but gradient descent on $E$ — and we can use the standard `torch.optim.SGD`, exactly as we trained the MLP in Ch 1. No need to reuse your previous manual implementation of local gradients: just use `.backward` as shown in the test case above.

One detail matters:

- **The weights stay untouched during this phase.** After inference, the weights should have *no gradients* at all — if they do, you accidentally backpropped to them. (There is a standard way to temporarily "freeze" a bunch of parameters.)

The states are independent tensors (from `init_states`), so the loop itself is a plain, standard SGD training loop — nothing else is needed.

In [ ]:
# === EXERCISE ===
def inference(self, x, y, iters: int = 200, track: bool = False):
    """Run Phase 1 (inference): settle the states to minimise the energy.

    Args:
        x:     input tensor
        y:     target tensor
        iters: number of energy-minimisation steps
        track: if True, also return the energy history

    Returns:
        the final energy if track=False, else the list of energies per step.

    The energy E is just another loss and the states are just another set of
    parameters, so settling them is gradient descent on E — exactly as you
    trained the MLP in Ch 1. Two things to remember:

    - Phase 1 holds the WEIGHTS fixed: set requires_grad_(False) on them
      before the loop and restore it afterwards, so inference never
      backprops to them.
    - The states are independent tensors (created in init_states), so the
      loop is a plain, standard SGD training loop: zero the state gradients,
      compute the energy, call E.backward(), take a step. Nothing else.
    """
    for p in self.parameters():
        p.requires_grad_(False)  # Phase 1: hold the weights fixed

    self.init_states(x)
    energies = []

    # YOUR CODE HERE
    raise NotImplementedError

    for p in self.parameters():
        p.requires_grad_(True)  # release the weights again
    return energies if track else E.item()


LinearPC.inference = inference

In [ ]:
# Test: does the settled energy reach the analytical optimum?
torch.manual_seed(4)
pc = LinearPC([2, 8, 1])

x1, y1 = X_train[:1], y_train[:1]
E_final = pc.inference(x1, y1, iters=300)
E_star = pc.analytic_E(x1, y1)

print(f"settled energy : {E_final:.6f}")
print(f"analytic  E*   : {E_star.item():.6f}")
print(f"difference     : {abs(E_final - E_star.item()):.2e}")

if abs(E_final - E_star.item()) < 1e-3:
    print("✓ test passed — inference converges to the analytical optimum!")
else:
    print("✗ not converged yet — try more iters or a smaller s_lr.")

# Phase 1 must hold the weights fixed: they should have NO gradients now.
weight_grads_clean = all(p.grad is None for layer in pc.layers for p in layer.parameters())
if weight_grads_clean:
    print("✓ test passed — the weights were held fixed during inference (no gradients)")
else:
    print("✗ inference backpropped to the weights — freeze them during inference!")

In [ ]:
# --- plotting: no need to read ---
# The "waiting for physics" visual: watch the energy settle toward E*.
history = pc.inference(x1, y1, iters=300, track=True)

plt.figure(figsize=(6, 4))
plt.plot(history, label="simulated energy E")
plt.axhline(E_star.item(), color="r", ls="--", label="analytic E*")
plt.xlabel("inference step")
plt.ylabel("E")
plt.title("Inference: energy settling")
plt.legend()
plt.show()

In [ ]:
# Also verify on a whole batch at once.
torch.manual_seed(5)
pc2 = LinearPC([2, 8, 1])
E_final_batch = pc2.inference(X_train[:32], y_train[:32], iters=300)
E_star_batch = pc2.analytic_E(X_train[:32], y_train[:32])
print(f"batch of 32: settled={E_final_batch:.6f} analytic={E_star_batch.item():.6f}")
assert abs(E_final_batch - E_star_batch.item()) < 1e-3, "inference should match E* on a batch too"
print("✓ batch test passed")

**Exercise 5 — `update_weights` (Phase 2).** Implement it: at the settled states, take one SGD step on the *weights*. The states must be treated as fixed.

Hint: to "freeze" a bunch of parameters, loop over them and set `requires_grad_(False)`, and set it back to `True` afterwards — the gradient then cannot flow into them, exactly as if they were `.detach()`ed.

In [ ]:
# === EXERCISE ===
def update_weights(self, x: torch.Tensor, y: torch.Tensor):
    """Phase 2: one gradient step on the weights, at the settled states.

    Returns the energy E after the update.

    The states were settled by inference and must be treated as FIXED here:
    freeze them (requires_grad_(False)) so the weight gradient cannot flow
    into them, and restore them afterwards. Also clear any gradients the
    states still carry from inference (s.grad = None). Then do one plain
    SGD step on the network parameters (self.parameters()) — E is just
    another loss.
    """
    # YOUR CODE HERE
    raise NotImplementedError


LinearPC.update_weights = update_weights

In [ ]:
# Test: update_weights holds the states fixed and updates the weights.
torch.manual_seed(6)
pc = LinearPC([2, 8, 1])
x1, y1 = X_train[:1], y_train[:1]
pc.inference(x1, y1, iters=50)

W_before = pc.layers[0].weight.detach().clone()
E = pc.update_weights(x1, y1)

states_clean = all(s.grad is None for s in pc.states[1:])
weights_moved = not torch.allclose(W_before, pc.layers[0].weight)

print(
    f"state gradients after update_weights: {[None if s.grad is None else 'SET' for s in pc.states[1:]]}"
)
print(f"weights changed: {weights_moved}")

if states_clean and weights_moved:
    print("✓ test passed — update_weights froze the states and moved the weights")
else:
    print("✗ check your implementation (states must be frozen during the weight step)")

Both phases are now implemented. Let's put them together: train the linear network for a few epochs and watch the energy go down. (The linear model can't separate the moons, but the *machinery* — settle the states, update the weights — is exactly what Ch 3 runs on the full network.)

In [ ]:
torch.manual_seed(8)
lin = LinearPC([2, 8, 1], s_lr=0.2, w_lr=0.02)
epochs = 20
energy_history = []
for epoch in range(epochs):
    perm = torch.randperm(len(X_train))
    epoch_E = 0.0
    n_batches = 0
    for i in range(0, len(X_train), 32):
        idx = perm[i : i + 32]
        lin.inference(X_train[idx], y_train[idx], iters=30)
        epoch_E += lin.update_weights(X_train[idx], y_train[idx])
        n_batches += 1
    energy_history.append(epoch_E / n_batches)
print(
    f"mean energy: epoch 1 = {energy_history[0]:.4f}  →  epoch {epochs} = {energy_history[-1]:.4f}"
)

In [ ]:
# --- plotting: no need to read ---
plt.figure(figsize=(6, 4))
plt.plot(energy_history)
plt.xlabel("epoch")
plt.ylabel("mean energy")
plt.title("Linear PC: energy over epochs")
plt.show()

# Ch 3 · Predictive Coding on the full network

**What we will explain here:** how everything from Ch 2 carries over unchanged to a nonlinear network, and how `torch.autograd` does the heavy lifting of the local chain rule through the nonlinearities.

## 3.1 Nonlinear PC on two-moons

Now we add the tanh activations back — the real network. There is no closed-form optimum anymore; we trust the machinery built and verified on the linear case.

Here is the nice part: **nothing changes.** The states, the energy, the two phases — all of it works out of the box, by inheritance. The only new ingredient is the nonlinearity, and that is where `E.backward()` earns its keep: it computes the *local* chain rule for us.

Recall the state dynamics from §2.2:

$$\frac{ds_i}{dt} = -\varepsilon_i + \left(\frac{\partial \hat s_{i+1}}{\partial s_i}\right)^\top \varepsilon_{i+1} .$$

The bottom-up term $-\varepsilon_i$ is trivial. The top-down term needs the Jacobian $(\partial \hat s_{i+1}/\partial s_i)^\top$: for a linear layer that is simply $W_{i+1}^\top$, but for $\hat s_{i+1} = \tanh(W_{i+1}s_i + b_{i+1})$ it becomes $W_{i+1}^\top \operatorname{diag}(1 - \tanh^2(z_{i+1}))$ — the weight matrix scaled by the derivative of the activation, exactly the kind of product `autograd` is built for. `E.backward()` computes all of this for us. The weight update keeps its clean local form,

$$\Delta W_i = -\eta\,\varepsilon_i\, s_{i-1}^\top ,$$

unchanged — the nonlinearity only changes how the *errors* are propagated during settling, not how the weights are updated.

Below is the full `PCNetwork` class. Note what it does: it *inherits* `energy`, `init_states`, `inference` and `update_weights` from `LinearPC` — everything you wrote in Ch 2 — and only adds the activated layers and the training step. That is the whole algorithm.

In [ ]:
class PCNetwork(LinearPC):
    """Predictive Coding network with tanh hidden layers and a sigmoid output.

    Inherits energy / init_states / inference / update_weights from LinearPC
    (all implemented in Ch 2), and adds the activated layers and the training
    step.
    """

    def __init__(self, layer_sizes: List[int], s_lr: float = 0.2, w_lr: float = 0.01):
        self.layer_sizes = layer_sizes
        self.s_lr = s_lr
        self.w_lr = w_lr
        self.layers = nn.ModuleList()
        for i in range(len(layer_sizes) - 1):
            if i < len(layer_sizes) - 2:
                self.layers.append(
                    nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh())
                )
            else:
                self.layers.append(
                    nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Sigmoid())
                )
        self.states = None
        self.errors = None

    def train_step(self, x: torch.Tensor, y: torch.Tensor, iters: int = 64) -> float:
        """One training step: Phase 1 (inference) then Phase 2 (weight update)."""
        self.inference(x, y, iters=iters)
        return self.update_weights(x, y)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        return self.y_hat(x)

In [ ]:
def train_pc(epochs=50, batch_size=32, iters=64, s_lr=0.2, w_lr=0.02):
    """Train a PC network on two-moons. Returns (net, energies, accs)."""
    torch.manual_seed(0)
    net = PCNetwork([2, 16, 16, 1], s_lr=s_lr, w_lr=w_lr)
    energies, accs = [], []

    for epoch in range(epochs):
        perm = torch.randperm(len(X_train))
        epoch_energy = 0.0
        n_batches = 0
        for i in range(0, len(X_train), batch_size):
            idx = perm[i : i + batch_size]
            epoch_energy += net.train_step(X_train[idx], y_train[idx], iters=iters)
            n_batches += 1
        energies.append(epoch_energy / n_batches)
        with torch.no_grad():
            acc = ((net.predict(X_test) > 0.5).float() == y_test).float().mean().item()
        accs.append(acc)

    return net, energies, accs


pc_net, pc_energies, pc_accs = train_pc()
print(f"Final test accuracy: {pc_accs[-1] * 100:.1f}%")

In [ ]:
# --- plotting: no need to read ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(pc_energies)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("mean energy")
axes[0].set_title("PC: training energy")
axes[1].plot(pc_accs)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test accuracy")
axes[1].set_title("PC: test accuracy")
plt.tight_layout()
plt.show()

In [ ]:
# --- plotting: no need to read ---
# The moment of truth: the same task, the same architecture —
# trained with backpropagation (Ch 1) and with Predictive Coding.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(mlp_model, X_train, y_train, ax=axes[0], title="Backpropagation")
plot_decision_boundary(pc_net, X_train, y_train, ax=axes[1], title="Predictive Coding")
plt.tight_layout()
plt.show()

In [ ]:
# --- plotting: no need to read ---
# The "waiting for physics" made visible: one test sample settling.
x_s, y_s = X_test[:1], y_test[:1]
settle = pc_net.inference(x_s, y_s, iters=150, track=True)

plt.figure(figsize=(6, 4))
plt.plot(settle)
plt.xlabel("inference step")
plt.ylabel("E")
plt.title("One test sample: states settling during inference")
plt.show()

## 3.2 ff_init — a digital-simulation trick

People struggled to simulate PC on computers: inference takes many steps, so training is slow. They invented tricks to make PC fit the hardware it was never intended for — a clocked, sequential digital machine.

The easy trick is **feedforward initialisation (`ff_init`)**: instead of starting the states at zero, run one quick forward pass and initialise the states to those feedforward values. The system then starts close to equilibrium, so settling is fast.

Because Ch 2 forced you to write an explicit `init_states` method, we can now simply **override** it.

A nuance for the analog picture: in analog hardware the states would not settle "instantly" — they would still *settle*, but the timescale is set by the physics (e.g. the RC constant of the circuit), not by a numerical iteration count. The iterative settling of a simulation is a software artefact. And note: computational tricks like `ff_init` are generally *not implementable physically* — that is fine for a computational trick, but good to keep in mind.

**Exercise.** Override `init_states` so that states start from a feedforward pass instead of zeros. The states are now *computed from the weights*: cut that dependency (`.detach()`) and re-enable `requires_grad`, so the weight update in `update_weights` stays local (see §3.1).

In [ ]:
# === EXERCISE ===
class PCNetworkFFInit(PCNetwork):
    """PC network with feedforward initialisation (ff_init)."""

    def init_states(self, x: torch.Tensor) -> None:
        """Initialise the states by a quick feedforward pass.

        Instead of zeros, set s_i to the layer-i output of the forward pass
        (this makes the hidden prediction errors ~0 at t=0). Keep s_0 = x.
        The states are computed from the weights here, so cut that dependency
        (e.g. with .detach()) and re-enable requires_grad — otherwise the
        weight update would backprop through them.
        """
        self.states = [x]
        s = x
        for i, layer in enumerate(self.layers[:-1]):
            # YOUR CODE HERE
            raise NotImplementedError

In [ ]:
# Quick training so this chapter is self-contained.
torch.manual_seed(7)
base = PCNetwork([2, 16, 16, 1], s_lr=0.2, w_lr=0.02)
for epoch in range(25):
    perm = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 32):
        idx = perm[i : i + 32]
        base.train_step(X_train[idx], y_train[idx], iters=64)

In [ ]:
# Compare zero-init vs ff_init on the same (trained) network.
zero = PCNetwork([2, 16, 16, 1], s_lr=0.2, w_lr=0.02)
ff = PCNetworkFFInit([2, 16, 16, 1], s_lr=0.2, w_lr=0.02)
zero.layers = base.layers
ff.layers = base.layers

x_s, y_s = X_test[:1], y_test[:1]
h_zero = zero.inference(x_s, y_s, iters=200, track=True)
h_ff = ff.inference(x_s, y_s, iters=200, track=True)

print(f"zero-init final E: {h_zero[-1]:.6f}")
print(f"ff-init   final E: {h_ff[-1]:.6f}")
print(f"energy after 10 steps — zero: {h_zero[9]:.4f}   ff: {h_ff[9]:.4f}")

same_fixed_point = abs(h_zero[-1] - h_ff[-1]) < 1e-3
if same_fixed_point and h_ff[9] < h_zero[9]:
    print("✓ test passed — same fixed point, and ff_init gets there much faster!")
else:
    print("✗ check your override — the two initialisations should meet the same fixed point.")

In [ ]:
# --- plotting: no need to read ---
plt.figure(figsize=(6, 4))
plt.plot(h_zero, label="zero initialisation")
plt.plot(h_ff, label="ff_init")
plt.xlabel("inference step")
plt.ylabel("E")
plt.legend()
plt.title("Zero init vs ff_init on a trained network")
plt.show()

### PC for digital simulation: ePC — error-based Predictive Coding

There is an even better computational trick, called **ePC** (error-based PC): instead of optimising the *states*, optimise the *errors* directly. This is **our own work** — Goemaere, Oliviers, Bogacz & Demeester, *"ePC: Fast and Deep Predictive Coding in Digital Simulation"* (arXiv:2505.20137, ICML 2026). It is again a trick with no physical equivalent, but it works *extremely* well.

The paper explains why PC simulations are notoriously slow: the standard state-based formulation (sPC) is (unfortunately by design) very difficult to simulate. So the slowness is a **hardware–algorithm mismatch**: a problem of the *simulation*, not of the algorithm itself.

All of these tricks — `ff_init`, `ePC` — try to make PC run well on computers. On analog hardware, they are unnecessary.

# Ch 4 · Takeaways

**What we will explain here:** the core argument of the thesis — why BP is a natural fit for von-Neumann machines, and why PC is a natural fit for analog hardware.

**Why BP is perfect for von-Neumann machines.** Sequential instruction streams, a global clock, cheap shared memory, a global loss broadcast. BP's global backward pass, stored activations, and layer locking map naturally onto that hardware.

**Why PC is perfect for analog hardware.**

- *No clock:* continuous-time settling; the circuit just relaxes, at its own RC timescale.
- *No memory:* no stored activations; errors are computed locally, on the fly, by neighbouring nodes. The energy is "computed by the physics".
- *No scheduling / layer locking:* all layers settle simultaneously; there is no reverse pass to schedule.
- *Just waiting for physics:* inference = letting the circuit find its energy minimum.

**The final translation table:**

| | Backpropagation | Predictive Coding |
|---|---|---|
| error signal | global, transported backward | local, emerges from settling |
| activations | stored (memory!) | states, transiently settled |
| scheduling | forward then reverse, layer by layer | all layers simultaneously |
| hardware | sequential, clocked, global | parallel, unclocked, local |

**Think.**

1. Could the PC algorithm of Ch 3 run without a global clock or scheduler?

<details><summary>Hint</summary>Look at the two phases: does the weight update need to know that inference has finished?</details>

<details><summary>Answer</summary>It *does* need a clock: something must separate the two phases — the weight update has to wait until the states have settled. That phase separation is arguably the final "show stopper" for a fully unclocked PC. (There are formulations in the literature that fuse the two phases into a single local dynamics, so it is not a fundamental obstacle — but it is worth knowing about.)</details>

2. If the node voltages of a circuit *are* the states, what must each node know to update itself, and how hard is that to wire?

<details><summary>Hint</summary>Translate "state, prediction from below, error from above" into circuit quantities: a voltage, a locally computed prediction, a difference.</details>

<details><summary>Answer</summary>Each node needs only local quantities: its own voltage, the prediction computed from the layer below, and the error from the layer above — all available from neighbouring nodes as voltages and currents, with subtraction naturally done by a differential pair. This is a **major advantage for circuit design**: everything stays locally connected with short wires, and no global bus is needed. The real engineering challenges are the top-down (error) connections and keeping the physical energy landscape well-behaved.</details>